# Agentic quant experiment analysis

This notebook reads the JSONL outputs produced by the framework and computes useful metrics: accuracy, runtime, estimated tokens, agent agreement, tie rate, and debate rounds.

In [ ]:
from pathlib import Path
import json
from collections import defaultdict
from statistics import mean

RESULTS_PATH = Path('../runs/latest/results.jsonl')
records = [json.loads(line) for line in RESULTS_PATH.open('r', encoding='utf-8') if line.strip()]
len(records), records[0].keys() if records else None

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd:
    df = pd.DataFrame(records)
    display(df.head())
else:
    for row in records[:3]:
        print(row['benchmark'], row['item_id'], row['mechanism'], row['final_answer'], row['correct'])

In [ ]:
def summarize(rows):
    groups = defaultdict(list)
    for row in rows:
        groups[(row['benchmark'], row['mechanism'])].append(row)
    summary = []
    for (benchmark, mechanism), items in sorted(groups.items()):
        summary.append({
            'benchmark': benchmark,
            'mechanism': mechanism,
            'n': len(items),
            'accuracy': mean(1 if item['correct'] else 0 for item in items),
            'mean_elapsed_s': mean(item['elapsed_s'] for item in items),
            'mean_total_tokens': mean(item['total_tokens'] for item in items),
            'mean_agreement': mean(item['agreement'] for item in items),
            'tie_rate': mean(1 if item['tie'] else 0 for item in items),
            'mean_debate_rounds': mean(item['debate_rounds'] for item in items),
        })
    return summary

summary = summarize(records)
if pd:
    summary_df = pd.DataFrame(summary)
    display(summary_df)
else:
    for row in summary:
        print(row)

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if pd and plt:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    summary_df.pivot(index='mechanism', columns='benchmark', values='accuracy').plot(kind='bar', ax=axes[0], ylim=(0, 1), title='Accuracy')
    summary_df.pivot(index='mechanism', columns='benchmark', values='mean_elapsed_s').plot(kind='bar', ax=axes[1], title='Mean time (s)')
    summary_df.pivot(index='mechanism', columns='benchmark', values='mean_total_tokens').plot(kind='bar', ax=axes[2], title='Mean total tokens')
    for ax in axes:
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
else:
    print('Install pandas and matplotlib for charts: python -m pip install -e ".[analysis]"')

In [ ]:
wrong = [row for row in records if not row['correct']]
print(f'Errors: {len(wrong)} / {len(records)}')
if pd and wrong:
    display(pd.DataFrame(wrong)[['benchmark', 'item_id', 'mechanism', 'final_answer', 'gold_answer', 'agreement', 'total_tokens']])
elif wrong:
    for row in wrong:
        print(row['benchmark'], row['item_id'], row['mechanism'], 'pred=', row['final_answer'], 'gold=', row['gold_answer'])

In [ ]:
agent_rows = []
for row in records:
    for agent_output in row['agent_outputs']:
        agent_rows.append({
            'benchmark': row['benchmark'],
            'mechanism': row['mechanism'],
            'item_id': row['item_id'],
            'agent': agent_output['agent'],
            'answer': agent_output['answer'],
            'confidence': agent_output['confidence'],
            'elapsed_s': agent_output['elapsed_s'],
            'tokens': agent_output['prompt_tokens'] + agent_output['completion_tokens'],
        })

if pd:
    agent_df = pd.DataFrame(agent_rows)
    display(agent_df.groupby(['mechanism', 'agent']).agg(mean_confidence=('confidence', 'mean'), mean_tokens=('tokens', 'mean'), mean_elapsed_s=('elapsed_s', 'mean')).reset_index())
else:
    print(agent_rows[:5])